In [1]:
import json
import os
from datetime import datetime

# File constants for data persistence
PATIENTS_FILE = "patients.txt"
VITALS_FILE = "vitals.txt"
APPOINTMENTS_FILE = "appointments.txt"

def load_data(filename, default_value):
    """Loads JSON data from a text file with error handling for missing/empty files."""
    if not os.path.exists(filename):
        return default_value
    try:
        with open(filename, "r") as file:
            content = file.read().strip()
            if not content:
                return default_value
            return json.loads(content)
    except (FileNotFoundError, json.JSONDecodeError):
        print(f"Warning: Could not read '{filename}'. Starting with default empty data.")
        return default_value


def save_data(filename, data):
    """Saves data structure to a text file using JSON formatting."""
    try:
        with open(filename, "w") as file:
            json.dump(data, file, indent=4)
        return True
    except IOError as e:
        print(f"Error saving data to {filename}: {e}")
        return False


# -------------------------------------------------------------------
# Input Validation Helpers
# -------------------------------------------------------------------

def get_valid_int(prompt, min_val=None, max_val=None):
    """Prompts for and validates an integer input within an optional range."""
    while True:
        try:
            val = int(input(prompt).strip())
            if min_val is not None and val < min_val:
                print(f"Value must be at least {min_val}.")
                continue
            if max_val is not None and val > max_val:
                print(f"Value must be at most {max_val}.")
                continue
            return val
        except ValueError:
            print("Invalid input! Please enter a valid integer.")


def get_valid_float(prompt, min_val=None, max_val=None):
    """Prompts for and validates a floating-point number input."""
    while True:
        try:
            val = float(input(prompt).strip())
            if min_val is not None and val < min_val:
                print(f"Value must be at least {min_val}.")
                continue
            if max_val is not None and val > max_val:
                print(f"Value must be at most {max_val}.")
                continue
            return val
        except ValueError:
            print("Invalid input! Please enter a valid numerical value.")


# -------------------------------------------------------------------
# Module 1: Patient Registration & Records
# -------------------------------------------------------------------

def add_patient(patients):
    """Registers a new patient with personal and health details."""
    print("\n--- Add New Patient ---")
    name = input("Enter Patient Name: ").strip().title()

    if not name:
        print("Error: Patient name cannot be empty.")
        return

    if name in patients:
        print(f"Patient '{name}' already exists in the system.")
        return

    age = get_valid_int("Enter Age: ", min_val=0, max_val=120)
    gender = input("Enter Gender (Male/Female/Other): ").strip().capitalize()
    contact = input("Enter Contact Number: ").strip()

    raw_conditions = input("Enter Health Conditions (comma-separated, e.g., Diabetes, BP): ")
    conditions = [cond.strip() for cond in raw_conditions.split(",") if cond.strip()]

    patients[name] = {
        "age": age,
        "gender": gender,
        "contact": contact,
        "conditions": conditions
    }
    print(f"Patient '{name}' registered successfully!")


def search_patient(patients):
    """Searches and displays a specific patient's record."""
    print("\n--- Search Patient ---")
    name = input("Enter Patient Name to search: ").strip().title()

    if name in patients:
        p = patients[name]
        print(f"\nRecord Found for '{name}':")
        print(f"  Age: {p['age']}")
        print(f"  Gender: {p['gender']}")
        print(f"  Contact: {p['contact']}")
        print(f"  Health Conditions: {', '.join(p['conditions']) if p['conditions'] else 'None'}")
    else:
        print(f"Error: Patient '{name}' not found.")


def update_patient(patients):
    """Updates contact or health condition details for an existing patient."""
    print("\n--- Update Patient Details ---")
    name = input("Enter Patient Name to update: ").strip().title()

    if name not in patients:
        print(f"Error: Patient '{name}' does not exist.")
        return

    p = patients[name]
    print(f"Updating records for {name}. Press Enter to keep current value.")

    age_str = input(f"Enter Age [{p['age']}]: ").strip()
    if age_str:
        try:
            p['age'] = int(age_str)
        except ValueError:
            print("Invalid age entered. Retaining previous age.")

    gender = input(f"Enter Gender [{p['gender']}]: ").strip()
    if gender:
        p['gender'] = gender.capitalize()

    contact = input(f"Enter Contact Number [{p['contact']}]: ").strip()
    if contact:
        p['contact'] = contact

    raw_cond = input(f"Enter Conditions (comma-separated) [{', '.join(p['conditions'])}]: ").strip()
    if raw_cond:
        p['conditions'] = [c.strip() for c in raw_cond.split(",") if c.strip()]

    print(f"Patient record for '{name}' updated successfully!")


def delete_patient(patients, vitals, appointments):
    """Deletes a patient record along with their associated vitals and appointments."""
    print("\n--- Delete Patient ---")
    name = input("Enter Patient Name to delete: ").strip().title()

    if name not in patients:
        print(f"Error: Patient '{name}' not found.")
        return

    confirm = input(f"Are you sure you want to delete patient '{name}'? (y/n): ").strip().lower()
    if confirm == 'y':
        del patients[name]
        if name in vitals:
            del vitals[name]

        # Remove active appointments for this patient
        appointments[:] = [app for app in appointments if app['patient_name'] != name]

        print(f"Patient '{name}' and associated records removed successfully.")


def view_all_patients(patients):
    """Displays all registered patient records."""
    print("\n--- All Registered Patients ---")
    if not patients:
        print("No patient records found.")
        return

    for name, info in patients.items():
        cond_str = ", ".join(info['conditions']) if info['conditions'] else "None"
        print(f"- Name: {name} | Age: {info['age']} | Gender: {info['gender']} | "
              f"Contact: {info['contact']} | Conditions: {cond_str}")


# -------------------------------------------------------------------
# Module 2: Patient Vitals Tracking
# -------------------------------------------------------------------

def track_vitals(patients, vitals):
    """Records daily vitals for a registered patient with current date stamping."""
    print("\n--- Track Patient Vitals ---")
    name = input("Enter Patient Name: ").strip().title()

    if name not in patients:
        print(f"Error: Patient '{name}' is not registered. Please register patient first.")
        return

    temp = get_valid_float("Enter Temperature (°F): ", min_val=80.0, max_val=115.0)
    sys_bp = get_valid_int("Enter Systolic Blood Pressure (mmHg): ", min_val=50, max_val=250)
    dia_bp = get_valid_int("Enter Diastolic Blood Pressure (mmHg): ", min_val=30, max_val=150)
    bp = f"{sys_bp}/{dia_bp}"
    heart_rate = get_valid_int("Enter Heart Rate (bpm): ", min_val=30, max_val=220)
    weight = get_valid_float("Enter Weight (kg): ", min_val=1.0, max_val=300.0)

    current_date = datetime.now().strftime("%Y-%m-%d %H:%M")

    vital_entry = {
        "date": current_date,
        "temperature": temp,
        "bp": bp,
        "heart_rate": heart_rate,
        "weight": weight
    }

    if name not in vitals:
        vitals[name] = []

    vitals[name].append(vital_entry)
    print(f"Vitals recorded for '{name}' on {current_date}!")


def view_vitals_history(vitals):
    """Displays historical vitals entries for a specific patient."""
    print("\n--- View Vitals History ---")
    name = input("Enter Patient Name: ").strip().title()

    if name not in vitals or not vitals[name]:
        print(f"No vitals history found for patient '{name}'.")
        return

    print(f"\nVitals History for {name}:")
    for idx, entry in enumerate(vitals[name], start=1):
        print(f"  [{idx}] Date: {entry['date']} | Temp: {entry['temperature']}°F | "
              f"BP: {entry['bp']} | HR: {entry['heart_rate']} bpm | Weight: {entry['weight']} kg")


# -------------------------------------------------------------------
# Module 3: Appointment Booking System
# -------------------------------------------------------------------

def book_appointment(patients, appointments):
    """Books a clinic appointment while checking for slot scheduling conflicts."""
    print("\n--- Book Appointment ---")
    name = input("Enter Patient Name: ").strip().title()

    if name not in patients:
        print(f"Warning: Patient '{name}' is not currently registered.")

    doctor = input("Enter Doctor Name: ").strip().title()
    date_str = input("Enter Appointment Date (YYYY-MM-DD): ").strip()
    time_str = input("Enter Appointment Time (e.g., 10:00 AM, 14:30): ").strip()

    # Check for slot availability conflict (Same Doctor at Same Date & Time)
    for app in appointments:
        if app['doctor_name'].lower() == doctor.lower() and \
           app['date'] == date_str and \
           app['time'].lower() == time_str.lower():
            print(f"Conflict Error: Dr. {doctor} is already booked at {time_str} on {date_str}.")
            return

    new_app = {
        "patient_name": name,
        "doctor_name": doctor,
        "date": date_str,
        "time": time_str
    }

    appointments.append(new_app)
    print(f"Appointment booked for {name} with Dr. {doctor} on {date_str} at {time_str}!")


def view_appointments(appointments):
    """Displays all booked clinic appointments."""
    print("\n--- All Appointments ---")
    if not appointments:
        print("No active appointments found.")
        return

    for idx, app in enumerate(appointments, start=1):
        print(f"  [{idx}] Patient: {app['patient_name']} | Doctor: {app['doctor_name']} | "
              f"Date: {app['date']} | Time: {app['time']}")


def cancel_appointment(appointments):
    """Cancels an existing appointment by patient name and date/time."""
    print("\n--- Cancel Appointment ---")
    if not appointments:
        print("No appointments available to cancel.")
        return

    name = input("Enter Patient Name: ").strip().title()
    date_str = input("Enter Appointment Date (YYYY-MM-DD): ").strip()

    matching = [app for app in appointments if app['patient_name'].lower() == name.lower() and app['date'] == date_str]

    if not matching:
        print(f"No appointments found for '{name}' on {date_str}.")
        return

    print("\nMatching Appointments Found:")
    for idx, app in enumerate(matching, start=1):
        print(f"  [{idx}] Doctor: {app['doctor_name']} | Time: {app['time']}")

    choice = get_valid_int("Select appointment number to cancel: ", min_val=1, max_val=len(matching))
    selected_app = matching[choice - 1]

    appointments.remove(selected_app)
    print(f"Appointment for {name} with Dr. {selected_app['doctor_name']} cancelled successfully.")


# -------------------------------------------------------------------
# Module 4 & 6: Main System Loop & Menu Navigation
# -------------------------------------------------------------------

def main():
    # Automatic Data Loading on Startup
    patients = load_data(PATIENTS_FILE, default_value={})
    vitals = load_data(VITALS_FILE, default_value={})
    appointments = load_data(APPOINTMENTS_FILE, default_value=[])

    while True:
        print("\n==============================================")
        print(" HEALTH MONITORING & APPOINTMENT SYSTEM ")
        print("==============================================")
        print("1. Add Patient")
        print("2. Update Patient")
        print("3. View Patient Records")
        print("4. Track Patient Vitals")
        print("5. View Vitals History")
        print("6. Book Appointment")
        print("7. View Appointments")
        print("8. Cancel Appointment")
        print("9. Save & Exit")
        print("==============================================")

        try:
            choice = input("Enter your choice (1-9): ").strip()

            if choice == '1':
                add_patient(patients)
            elif choice == '2':
                update_patient(patients)
            elif choice == '3':
                print("\n  a. View All Records\n  b. Search Patient\n  c. Delete Patient")
                sub_choice = input("Select option (a/b/c): ").strip().lower()
                if sub_choice == 'a':
                    view_all_patients(patients)
                elif sub_choice == 'b':
                    search_patient(patients)
                elif sub_choice == 'c':
                    delete_patient(patients, vitals, appointments)
                else:
                    print("Invalid option selected.")
            elif choice == '4':
                track_vitals(patients, vitals)
            elif choice == '5':
                view_vitals_history(vitals)
            elif choice == '6':
                book_appointment(patients, appointments)
            elif choice == '7':
                view_appointments(appointments)
            elif choice == '8':
                cancel_appointment(appointments)
            elif choice == '9':
                save_data(PATIENTS_FILE, patients)
                save_data(VITALS_FILE, vitals)
                save_data(APPOINTMENTS_FILE, appointments)
                print("All data saved successfully. Existing program. Goodbye!")
                break
            else:
                print("Invalid menu selection! Please choose a number between 1 and 9.")

        except KeyboardInterrupt:
            print("\n\nSession interrupted. Saving data before exit...")
            save_data(PATIENTS_FILE, patients)
            save_data(VITALS_FILE, vitals)
            save_data(APPOINTMENTS_FILE, appointments)
            break
        except Exception as e:
            print(f"An unexpected error occurred: {e}")


if __name__ == "__main__":
    main()


 HEALTH MONITORING & APPOINTMENT SYSTEM 
1. Add Patient
2. Update Patient
3. View Patient Records
4. Track Patient Vitals
5. View Vitals History
6. Book Appointment
7. View Appointments
8. Cancel Appointment
9. Save & Exit
Enter your choice (1-9): 7

--- All Appointments ---
No active appointments found.

 HEALTH MONITORING & APPOINTMENT SYSTEM 
1. Add Patient
2. Update Patient
3. View Patient Records
4. Track Patient Vitals
5. View Vitals History
6. Book Appointment
7. View Appointments
8. Cancel Appointment
9. Save & Exit
Enter your choice (1-9): 9
All data saved successfully. Existing program. Goodbye!
